In [9]:
import cv2
import numpy as np
import rasterio
from rasterio.features import shapes
import numpy as np
from skimage import measure, morphology
from shapely.geometry import shape, mapping
import fiona
import json

input_img = "input.png"
binary_img = "binary_map.png" #intermediate operation, don't need change
bbox_geojson = "bounding_box.json" # four corner coordinates in order: top-left, top-right, bottom-right, bottom-left
threshold = 0.5                     # if probability raster
simplify_tolerance = 2.0            # units in arbitrary coordinate system(meters) *  should change implementation to proportion of image size
min_area_pixels = 20
output_geojson = "output.geojson"                # remove small noise * should change implementation to proportion of image size

In [10]:
# Load image
img = cv2.imread(input_img)  # BGR format in OpenCV
b, g, r = cv2.split(img) # Split into channels

# Define threshold for red detection
threshold = 100  # tweak as needed

# Create binary mask: 1 if red is strong and green/blue are weak
binary_map = np.where((r > threshold) & (g < threshold) & (b < threshold), 1, 0)

# Optional: save as image
cv2.imwrite("binary_map.png", (binary_map * 255).astype(np.uint8))

True

In [16]:
with open(bbox_geojson) as f: #bounding box geojson input from webapp
    bbox_data = json.load(f)

bbox_geom = shape(bbox_data)  # convert to shapely Polygon
min_lon, min_lat, max_lon, max_lat = bbox_geom.bounds

with rasterio.open(binary_img) as src: #buildings binary image
    raster = src.read(1)
height, width = raster.shape
print(height, width)

from rasterio.transform import from_bounds
transform = from_bounds(
    min_lon, min_lat,  # west, south
    max_lon, max_lat,  # east, north
    width, height      # your raster or image dimensions in pixels
)

634 722


c:\Users\tanle\Documents\GitHub\Segmentation\.venv\Lib\site-packages\rasterio\__init__.py:368: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


In [17]:
# --------------------------
# Threshold if raster is soft probability
# --------------------------
mask = raster > threshold

# Remove small objects (noise)
mask = morphology.remove_small_objects(mask.astype(bool), min_size=min_area_pixels)

# Label/tag/number connected components
labels = measure.label(mask)

# Polygonize each connected component: converting connected pixel regions in a binary mask into vector polygons
polygons = []
for region in measure.regionprops(labels):
    coords = region.coords
    single_mask = np.zeros_like(mask, dtype=np.uint8)
    single_mask[tuple(coords.T)] = 1
    for geom, val in shapes(single_mask, mask=single_mask, transform=transform):
        if val == 1:
            poly = shape(geom) #turns a pixel-wise raster outline into a polygon
            poly = poly.simplify(simplify_tolerance, preserve_topology=True) #simplify the polygon shape, reduce vertex count
            polygons.append(poly)

# --------------------------
# Remove empty geometries
# --------------------------
polygons = [poly for poly in polygons if poly.area > 0]

In [18]:
print(polygons)

[<POLYGON ((103.822 1.469, 103.821 1.468, 103.822 1.468, 103.822 1.469))>, <POLYGON ((103.828 1.467, 103.827 1.466, 103.829 1.465, 103.829 1.466, 103.8...>]


In [19]:
# --------------------------
# Save to GeoJSON (EPSG:4326)
# --------------------------
geojson_features = []
for idx, poly in enumerate(polygons):
    geojson_features.append({
        "type": "Feature",
        "geometry": mapping(poly),
        "properties": {"id": idx}
    })

geojson_dict = {
    "type": "FeatureCollection",
    "crs": {"type": "name", "properties": {"name": "EPSG:4326"}},
    "features": geojson_features
}

with open(output_geojson, "w") as f:
    json.dump(geojson_dict, f)

print(f"✅ Polygonized buildings saved to {output_geojson} in EPSG:4326 coordinates.")

✅ Polygonized buildings saved to output.geojson in EPSG:4326 coordinates.


In [20]:
print(geojson_dict)

{'type': 'FeatureCollection', 'crs': {'type': 'name', 'properties': {'name': 'EPSG:4326'}}, 'features': [{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': (((103.82208114153444, 1.46863182241776), (103.82137300530455, 1.4683900591052976), (103.82161480791963, 1.4682001022169344), (103.82208114153444, 1.46863182241776)),)}, 'properties': {'id': 0}}, {'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': (((103.82807439206549, 1.4666804471100285), (103.82722808291268, 1.4656270498200141), (103.82859254052639, 1.4653334800834528), (103.82888615798755, 1.466283264525269), (103.82807439206549, 1.4666804471100285)), ((103.82817802175767, 1.466335070949368), (103.82821256498839, 1.466352339757401), (103.82821256498839, 1.466335070949368), (103.82817802175767, 1.466335070949368)))}, 'properties': {'id': 1}}]}
